In [ ]:
import pandas as pd

## New Topic Classification

In [ ]:
train_df = pd.read_csv('/content/drive/MyDrive/MLDatasets/text_classification/train_10k.csv')
train_df.head()

,label,heading,text
0,2,Wall St. Bears Claw Back Into the Black (Reuters),"Reuters - Short-sellers, Wall Street's dwindli..."
1,2,Carlyle Looks Toward Commercial Aerospace (Reu...,Reuters - Private investment firm Carlyle Grou...
2,2,Oil and Economy Cloud Stocks' Outlook (Reuters),Reuters - Soaring crude prices plus worries\ab...
3,2,Iraq Halts Oil Exports from Main Southern Pipe...,Reuters - Authorities have halted oil export\f...
4,2,"Oil prices soar to all-time record, posing new...","AFP - Tearaway world oil prices, toppling reco..."


In [ ]:
train_df.label.value_counts()

,count
label,
2,10000
3,10000
1,10000
0,10000


In [ ]:
label_encoding = {0:'world',1:'sports',2:'business',3:'science_tech'}

## Build the vocabulary

In [ ]:
import re
_token_reg = re.compile(r"[a-z0-9]+")
# this regex will tokenize the text into alphanumeric tokens


def tokenize(text):
  text=text.lower()
  tokens = _token_reg.findall(text)
  return tokens


In [ ]:
print(train_df.iloc[0].text)

tokens = tokenize(train_df.iloc[0].text)
print(tokens)

Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.
['reuters', 'short', 'sellers', 'wall', 'street', 's', 'dwindling', 'band', 'of', 'ultra', 'cynics', 'are', 'seeing', 'green', 'again']


In [ ]:
from collections import Counter

def build_vocab(texts,max_size=50000):
  counter = Counter()
  for text in texts:
    tokens = tokenize(text)
    counter.update(tokens)

  return counter

# counter takes all tokens of corpus as input and computes their frequencies
# it counts how many times a given token has appeared in given corpus

In [ ]:
counter = build_vocab(train_df['text'].to_list())

In [ ]:
len(counter)

38029

In [ ]:
len(counter.most_common(30000))
## most common method helps to filter the most occuring tokens for building vocab

30000

In [ ]:
class Vocab:
  def __init__(self,counter,max_vocab_size):
    limited_counter = counter.most_common(max_vocab_size)

    self.idxTotoken=[]
    for token,freq in limited_counter:
      self.idxTotoken.append(token)

    self.idxTotoken.append('<pad>')
    self.idxTotoken.append('<unk>')

    self.tokenToidx = {}

    for idx,token in enumerate(self.idxTotoken):
      self.tokenToidx[token]=idx

    self.pad_token_idx = self.tokenToidx['<pad>']
    self.unk_token_idx = self.tokenToidx['<unk>']


  def __len__(self):
    return len(self.tokenToidx)

  def encode(self,tokens):
    token_ids = []
    for token in tokens:
      token_id = self.tokenToidx.get(token,self.unk_token_idx)
      # if token in not part of vocab - use unk token_id for that token
      token_ids.append(token_id)

    return token_ids


In [ ]:
vocab = Vocab(counter=counter,max_vocab_size=30000)

In [ ]:
len(vocab)

30002

In [ ]:
text = 'Paris is capital of France'
tokens = tokenize(text)
print(tokens)

['paris', 'is', 'capital', 'of', 'france']


In [ ]:
token_ids = vocab.encode(tokens)
print(token_ids)

[800, 17, 406, 3, 770]


In [ ]:
from torch.utils.data import Dataset,DataLoader
from torch.nn.utils.rnn import pad_sequence,pack_padded_sequence
import torch

In [ ]:
class TextDataset(Dataset):
  def __init__(self,texts,labels,vocab,max_len=256):
    self.vocab=vocab
    self.max_len=max_len
    self.datapoints = []
    for text,label in zip(texts,labels):
      tokens = tokenize(text)[:self.max_len] #truncation
      token_ids = self.vocab.encode(tokens)
      self.datapoints.append((torch.tensor(token_ids,dtype=torch.long),label))

  def __len__(self):
    return len(self.datapoints)

  # this function is needed for working with indexing
  def __getitem__(self,idx):
    return self.datapoints[idx]



# applying padding for batches
# converting data into torch tensors
def collate_fn(pad_index:int):
  def collate(batch):
    sequences,labels = zip(*batch)
    lengths = torch.tensor([len(s) for s in sequences],dtype=torch.long)
    padded = pad_sequence(sequences,batch_first=True,padding_value=pad_index)
    labels = torch.tensor(labels,dtype=torch.long)
    return padded,lengths,labels
  return collate


# torch.tensor -> matrix datatypes in torch
# We are converting all matrixes in torch.long -> which is a float datatype in torch
# All matrixes need to be in same datatype for correct operations



## Why padding is needed ?
'''
text 1 : This is a great place
-> this is a great place
token_ids : t0 t1 t2 t3 t4

--padding
text 2 : Hello all
-> hello all <pad> <pad> <pad>
token_ids : t0 t1 t2 t3 t4


--truncation
text 3 : There is a very big waterfall
-> there is a very big
token_ids : t0 t1 t2 t3 t4

Now as the number of token ids are matching, these two texts can be passed in a single batch matrix
'''


'\ntext 1 : This is a great place\n-> this is a great place\ntoken_ids : t0 t1 t2 t3 t4\n\n--padding\ntext 2 : Hello all\n-> hello all <pad> <pad> <pad>\ntoken_ids : t0 t1 t2 t3 t4\n\n\n--truncation\ntext 3 : There is a very big waterfall\n-> there is a very big\ntoken_ids : t0 t1 t2 t3 t4\n\nNow as the number of token ids are matching, these two texts can be passed in a single batch matrix\n'

In [ ]:
vocab.unk_token_idx

30001

## RNN Model Architecture

In [ ]:
from torch import nn

class RNNTextClf(nn.Module):
  def __init__(self,vocab_size,embed_dim,hidden_dim,num_classes,pad_idx,num_layers=1,
               dropout=0.3):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size,embed_dim,padding_idx=pad_idx)
    self.lstm = nn.LSTM(
        input_size = embed_dim,
        hidden_size = hidden_dim, # number of nodes in LSTM layer
        batch_first = True,
        bidirectional = True,
        num_layers = num_layers,
        dropout = dropout if num_layers>1 else 0.0
    )
    self.dropout = nn.Dropout(dropout)

    # classification head - output layer
    self.clf_head = nn.Linear(in_features= hidden_dim*2, # *2 is due to bidirectional nature
                              out_features=num_classes)


  def forward(self,batch,lengths):
    # batch : (batch_size,seq_lengths)
    embedded = self.embedding(batch)
    embedded = self.dropout(embedded)

    packed = pack_padded_sequence(
        embedded,lengths.cpu(),batch_first=True,enforce_sorted=False
    )
    _,(hidden,_) = self.lstm(packed)

    # hidden : num_layers*2(bidirectional) x batch_size x hidden_dim
    # hidden[-2] : left to right context vector
    # hidden[-1] : right to left context vector
    concat_hidden = torch.concat((hidden[-2],hidden[-1]),dim=1)
    concate_hidden = self.dropout(concat_hidden)
    return self.clf_head(concat_hidden)


In [ ]:
## this is two in one loss function , which applies softmax on given inputs and then computes loss wrt labels
## Due to this we don't need a dedicated softmax activation in last layer of model

def run_epoch(model,loader,criterion,optimizer,device,is_train=True):
  model.train(is_train)
  total_loss = 0
  if is_train:
    print("Running Train Epoch ....")
  else:
    print("Running Test Epoch ....")
  with torch.set_grad_enabled(is_train):
    for i ,(padded,lengths,labels) in enumerate(loader):
      padded,lengths,labels = padded.to(device),lengths.to(device),labels.to(device)
      output = model(padded,lengths)
      loss = criterion(output,labels)
      # CELoss will first apply softmax on output and then compute loss wrt labels

      if is_train:
        optimizer.zero_grad() # clear older gradients
        loss.backward() #compute gradients using backprop
        optimizer.step() #update model weights

      total_loss+=loss.item()

  avg_loss = total_loss/len(loader)
  print(f"Average epoch loss : {avg_loss}")
  print("--"*40)

  return avg_loss



In [ ]:
test_df = pd.read_csv('/content/drive/MyDrive/MLDatasets/text_classification/test_2k.csv')
test_df.head()

,label,heading,text
0,2,Fears for T N pension after talks,Unions representing workers at Turner Newall...
1,3,The Race is On: Second Private Team Sets Launc...,"SPACE.com - TORONTO, Canada -- A second\team o..."
2,3,Ky. Company Wins Grant to Study Peptides (AP),AP - A company founded by a chemistry research...
3,3,Prediction Unit Helps Forecast Wildfires (AP),AP - It's barely dawn when Mike Fitzpatrick st...
4,3,Calif. Aims to Limit Farm-Related Smog (AP),AP - Southern California's smog-fighting agenc...


In [ ]:
train_df.isna().sum()

,0
label,0
heading,0
text,0


In [ ]:
train_df['text']=train_df['text'].apply(lambda x:x.strip())
train_df[train_df['text']=='']

,label,heading,text


In [ ]:
train_texts = train_df['text'].to_list()
train_labels = train_df['label'].to_list()

In [ ]:
test_df['text']=test_df['text'].apply(lambda x:x.strip())
test_df[test_df['text']=='']

,label,heading,text


In [ ]:
test_texts = test_df['text'].to_list()
test_labels = test_df['label'].to_list()

In [ ]:
## Create datasets
MAX_LEN=256
train_ds = TextDataset(train_texts,train_labels,vocab,max_len=MAX_LEN)
test_ds = TextDataset(test_texts,test_labels,vocab,max_len=MAX_LEN)

collate = collate_fn(vocab.pad_token_idx)

In [ ]:
train_ds[100]

(tensor([  298,   101,     1,  3714,   513,  1214,     2,  8461,  6214,     5,
         19384,   222,  1655,     2,    30,   167,     2,  3059,   708,     0,
         19385,  3060,    34,  1727,    57,  7024,     5,    89,    30,    12,
          2066,    12, 24457,    19,  9482,  3404]),
 3)

In [ ]:
BATCH_SIZE = 64
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,shuffle =True,collate_fn = collate)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE,shuffle =True,collate_fn = collate)

In [ ]:
batch = next(iter(train_loader))

In [ ]:
# token_ids
batch[0].shape

torch.Size([64, 56])

In [ ]:
# labels
batch[2].shape

torch.Size([64])

In [ ]:
len(train_loader)

625

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
# device = cuda is GPU
print(device)

cuda


In [ ]:
model = RNNTextClf(vocab_size=len(vocab),
                   embed_dim=128,
                   hidden_dim=256,
                   num_classes=4,
                   pad_idx=vocab.pad_token_idx,
                   num_layers=2,
                   dropout=0.3).to(device)

def train(model,
          lr=1e-3,n_epochs = 3,
          model_save_path='best_model.pt'):
  criterion = nn.CrossEntropyLoss()
  optimizer = torch.optim.Adam(model.parameters(),lr=lr)

  best_loss = 1e1000

  for epoch in range(n_epochs):
    train_loss = run_epoch(model,train_loader,criterion,optimizer,device,is_train=True)

    test_loss = run_epoch(model,test_loader,criterion,optimizer,device,is_train=False)

    if test_loss < best_loss:
      best_loss = test_loss
      # save the model
      torch.save(
          {
              "model_state": model.state_dict(),
              'vocab': vocab.idxTotoken,
              'label_encoding': label_encoding
          },
          model_save_path
      )

In [ ]:
train(model)

Running Train Epoch ....
Average epoch loss : 0.7294549302577973
--------------------------------------------------------------------------------
Running Test Epoch ....
Average epoch loss : 0.48817478646250334
--------------------------------------------------------------------------------
Running Train Epoch ....
Average epoch loss : 0.4181585536003113
--------------------------------------------------------------------------------
Running Test Epoch ....
Average epoch loss : 0.4235187675021276
--------------------------------------------------------------------------------
Running Train Epoch ....
Average epoch loss : 0.318889056789875
--------------------------------------------------------------------------------
Running Test Epoch ....
Average epoch loss : 0.3812476933753791
--------------------------------------------------------------------------------


In [ ]:
def predict(model,texts,vocab,label_encoding,max_len,device):

  sequences = []
  for t in texts:
    tokens = tokenize(t)[:max_len]
    token_ids = vocab.encode(tokens)
    sequences.append(torch.tensor(token_ids,dtype=torch.long))

  lengths = torch.tensor([len(s) for s in sequences],dtype=torch.long)
  padded = pad_sequence(sequences,batch_first=True,padding_value=vocab.pad_token_idx)

  model.eval() ## in eval mode no gradients are computed
  output = model(padded.to(device),lengths.to(device))
  probabilities = torch.softmax(output,dim=1)
  preds = probabilities.argmax(dim=1).cpu()

  for t,p in zip(texts,preds):
    print("Text:",t)
    print("Label:",label_encoding[int(p)])
    print("--"*60)

  return preds


In [ ]:
texts = test_df['text'].iloc[[120,400]].to_list()
texts

['The price of oil has continued its sharp rise overnight, closing at a record high. The main contract in New York, light sweet crude for delivery next month, has closed at a record \\$US46.75 a barrel - up 70 cents on yesterday #39;s close.',
 'It can only remain aloft for three minutes but weighs less than an empty soft drink can -- and it can take and transmit pictures in flight.']

In [ ]:
preds = predict(model,texts,vocab,label_encoding,256,device)
preds

Text: The price of oil has continued its sharp rise overnight, closing at a record high. The main contract in New York, light sweet crude for delivery next month, has closed at a record \$US46.75 a barrel - up 70 cents on yesterday #39;s close.
Label: business
------------------------------------------------------------------------------------------------------------------------
Text: It can only remain aloft for three minutes but weighs less than an empty soft drink can -- and it can take and transmit pictures in flight.
Label: science_tech
------------------------------------------------------------------------------------------------------------------------


tensor([2, 3])

In [ ]:
test_df.iloc[[120,400]]

,label,heading,text
120,0,Oil prices bubble to record high,The price of oil has continued its sharp rise ...
400,3,"Prototype copter-cam: Here, there, everywhere",It can only remain aloft for three minutes but...
